In [ ]:
using Combinatorics, ITensors, JLD2, Yao, ITensorMPS, Plots, NPZ, ProgressMeter
include("../../src/apply_gate_as_mpo.jl");
include("../../src/gates_utils.jl");
#Script with operations w/ majoranas
include("../../src/majo_machinery.jl");
#Script with possible initial configurations
include("../../src/initial_circuits.jl");
#Script with possible unkown transformations
include("../../src/unknown_circuits.jl");
#Script with potential quantum gates
include("../../src/gates_circuit.jl");
#Script with possible hamiltonian evolutions
include("../../src/ham_evols.jl");

In [ ]:
@show Threads.nthreads()

In [ ]:
#number qubits 
nq = 55
#we want the products in module Bm
m = 2
#obtain majoranas and coefficients
list_majoranas = majorana_products(nq, m)
len_maj = length(list_majoranas);
#Operators out of list_majoranas
list_maj_op = [ops for (_, ops) in list_majoranas]
#Coefficients out of list_majoranas
coeff_maj_op = [real(coef) for (coef,_) in list_majoranas]
#Observable (XX pair middle qubits)
op_flo_ = [pauli_product(majorana_symbol(nq, 2*i), majorana_symbol(nq, 2*i + 1))[2] for i in 1:(nq - 1)]
random_coeffs = ones(length(op_flo_))
#random_coeffs = randn(length(op_flo_)) #uncomment to use random coefficients

@show op_flo_, random_coeffs

#op_flo_ = fill(:id, nq)
#for i in 1:(nq)
#    op_flo_ = pauli_product(op_flo_, majorana_symbol(nq, i))[2]
#end
#op_flo_

In [ ]:
# create folder named $(nq)q to save data
base_data_dir = "../data"
data_dir = joinpath(base_data_dir, "$(nq)q")
if !isdir(data_dir)
    mkdir(data_dir)
end

In [ ]:
#Function to perform multiplication coeff*op
function make_operator(sites::Vector{Index{Int64}}, op::Symbol, index::Int)
    ITensor(paulis[op], sites[index]', sites[index])
end

#Initialize sites for tensors
sites_maj = siteinds("Qubit", nq)

# Generate paulis out of majoranas
paulis_maj = [MPO([make_operator(sites_maj, list_maj_op[i][j], j)
               for j in 1:nq]) for i in 1:len_maj];

# Include coeffcients to be actual majoranas, not just Paulis
op_maj = paulis_maj .* coeff_maj_op;


# Convert FLO into MPO
op_flo = [MPO([make_operator(sites_maj, op_flo_[i][j], j) for j in 1:nq]) for i in 1:length(op_flo_)];
op_flo = op_flo .* random_coeffs;

In [ ]:
npzwrite(joinpath(data_dir, "coeffs.npy"), random_coeffs)

In [ ]:
# Function to get contractions gate |> state

function get_state!(gates, state::MPS, sites::Vector{Index{Int}}, indices::Vector{Vector{Int}})
    final_state = copy(state)
    
    for (j, gate) in enumerate(gates)
        inds = indices[j]
        it = length(size(gate)) == 2 ?
             ITensor(gate, sites[inds[1]]', sites[inds[1]]) :
             ITensor(gate, sites[inds[1]]', sites[inds[2]]', sites[inds[1]], sites[inds[2]])

        final_state = noprime(apply_mpo_gate(final_state, gate_to_mpo(it), inds));
    end

    return final_state
end

In [ ]:
# Function to group observable in case there is a tradeoff in which increasing χ and reducing #operators is faster

function group_mpos(mpos::Vector{MPO}, size_g::Int; maxbond::Int)
    ng = ceil(Int, length(mpos) / size_g)
    grouped = Vector{MPO}(undef, ng)

    for i in 1:ng
        range = (i-1)*size_g + 1 : min(i*size_g, length(mpos))
        grouped[i] = truncate(sum(mpos[range]), maxdim = maxbond)
    end

    return grouped
end

In [ ]:
#Initialize zero state 
zero_state = fill("0",nq)
MPS_zero = productMPS(sites_maj, zero_state);

#how many points obtained as different angles t
num_points = 1001
t_list = collect(range(0,2π,num_points))
dt = t_list[2] - t_list[1]
println("dt = ", dt)
println("min/max = ", minimum(t_list), "/", maximum(t_list))
npzwrite(joinpath(data_dir, "t_list.npy"), t_list)

#Layer in quantum circuit
nlayers = 1

#Maximum bond dimension initial state
#If after given layer χ is greater than maxbond then truncate to χ and get state out of circuit
maxbond = 64


# Input states are squeeze state (One-Axis Twisting), i.e., H is sum of XX on all qubit paris
#Types of circuits for initialization
state_generators = [
    generate_OATH_dataset,
    generate_OATH_X_dataset,
    one_axis_twisting_x,
    random_rotations_with_entanglement_state_circuit,
    simple_extent_state_circuit,
    first_qubit_rotation_state_circuit,
    random_rotations_state_circuit,
    random_fermionic_gaussian_circuit
];

#Initial state after applying geenrator circuit 
#initial_state = state_generators[2](MPS_zero,sites_maj,t_list; std=0.1, use_two_ax=false, dt=dt, 
#verb = true);
# thread parallelization:
initial_state = [copy(MPS_zero) for _ in 1:length(t_list)]
@showprogress Threads.@threads for i in 1:length(t_list)
    initial_state[i] = state_generators[3](initial_state[i],sites_maj,t_list[i])
end

In [ ]:
# A fuction to compute individual overlaos: easier to track down

function calculate_contributions(op_list::Vector{MPO}, st::MPS)

    contributions = [noprime(op * st) for op in op_list];

    return contributions

end

In [ ]:
# Select observable that is full observable (up to chosen number of terms)

observable_flo = op_flo;

# Select observable that is the baiss elements in the module

observable_maj = op_maj;

In [ ]:
num_param_points = 41
@assert isodd(num_param_points) "num_param_points should be odd to have a point at theta=0"
Δθ = 0.001
θ₀ = 0.0
#θ_start = #θ₀ - ((num_param_points - 1) / 2) * Δθ
θ_list = collect(range(0.01, 0.2, num_param_points ÷ 2)) #10 .^ collect(range(-5, -1, num_param_points ÷ 2))
θ_list = [(-reverse(θ_list))..., θ₀, θ_list...]
@assert length(θ_list) == num_param_points
θ_start = θ_list[1]
# "Unknown" transformation (i.e., S_theta using e^i theta H /2 with H = sum_i Z_i for magnetometry)
@show(θ₀, θ_list)
U(psi, θ) = product_rotations(psi, sites_maj, θ, Pz)

In [ ]:
npzwrite(joinpath(data_dir, "theta_list.npy"), θ_list)

In [ ]:
# Compute outputs
outputs = zeros(Float64, num_points, num_param_points)

@showprogress Threads.@threads for i in 1:num_points
    transformed_state = copy(initial_state[i])
    for j in 1:(num_param_points)
        if j > 1
            transformed_state = U(transformed_state, θ_list[j] - θ_list[j-1])
        else
            transformed_state = U(transformed_state, θ_start)
        end
        outputs[i,j] = sum(real(inner(transformed_state', observable_flo[j], transformed_state)) for j in 1:length(observable_flo))
    end
end

In [ ]:
Plots.plot(t_list, outputs[:,num_param_points ÷ 2 + 1] - outputs[:,num_param_points ÷ 2 - 1], label = "Derivative / theta", xlabel = "t", ylabel = "Derivative", title = "Derivative vs t", legend = :topright, markershape = :circle, markercolor = :blue, linewidth = 2, grid = true)

In [ ]:
# Compute feature vectors

feat_vectors = Vector{Vector{Float64}}(undef, num_points)

@showprogress Threads.@threads for i in 1:num_points
    #println("Processing sample $i on thread $(Threads.threadid())")
    vec = Vector{Float64}(undef, len_maj)
    for j in 1:len_maj
        vec[j] = real(inner(initial_state[i], observable_maj[j], initial_state[i]))
    end
    feat_vectors[i] = vec
end

In [ ]:
npzwrite(joinpath(data_dir, "outputs.npy"), outputs)
array_data = reduce(vcat, feat_vectors')  # convert to 10×N matrix
npzwrite(joinpath(data_dir, "feat_vec.npy"), array_data)
